In [11]:
import pandas as pd
EXCLUDED_WHOLE = [23, 24, 25, 27]
DROPPED_INDICES = [10] + [1 + x for x in EXCLUDED_WHOLE]
df = pd.read_csv("question_artefacts/pfairness_questions - questions_only.csv")
complete_task_list_df = df[["qidx", "question_clean", "question_nickname", "dataset_file"]].drop(index=DROPPED_INDICES)
complete_task_list_df["qidx"] = complete_task_list_df["qidx"].astype(int)
complete_task_list_df.columns = complete_task_list_df.columns.str.replace('_', r'\\_', regex=True)

In [12]:
print(complete_task_list_df.replace('_', r'\\_', regex=True).to_latex(index=False))

\begin{tabular}{rlll}
\toprule
qidx & question\_clean & question\_nickname & dataset\_file \\
\midrule
0 & Sentiment refers to the emotional tone or attitude expressed in a piece of text, typically categorized as positive, negative,or neutral based on the conveyed feelings or opinions.

Example
"The address has been updated" has neutral sentiment.

You are using a language model like ChatGPT to determine the sentiment of a text [TEXT] into one of possible three sentiment labels: positive, negative, and neutral.

How would you phrase your request to the language model? Your response must include the placeholder [TEXT]. & Sentiment Analysis 3 Classes & 500-sentiment\_3class.csv \\
1 & Sentiment refers to the emotional tone or attitude expressed in a piece of text. In this task, the goal is to classify a text into one of two sentiment classes: positive or negative.

Example
"I love the iPhone" has positive sentiment.

You are using a language model like ChatGPT to determine the sentiment 

In [18]:
complete_task_metrics_df = df[["qidx", "question_nickname", "metric_name"]].drop(index=DROPPED_INDICES)
complete_task_metrics_df["qidx"] = complete_task_metrics_df["qidx"].astype(int)
complete_task_metrics_df.columns = complete_task_metrics_df.columns.str.replace('_', r'\\_', regex=True)
print(complete_task_metrics_df.replace('_', r'\\_', regex=True).replace('\\n', ", ", regex=True).to_latex(index=False))

\begin{tabular}{rll}
\toprule
qidx & question\_nickname & metric\_name \\
\midrule
0 & Sentiment Analysis 3 Classes & f1 \\
1 & Sentiment Analysis 2 Classes & f1 \\
2 & Machine Translation & bleu \\
3 & Dialogue Summarization & rouge \\
4 & Topic Classification & f1 \\
5 & Free Generation: Balonku Song & llm\_judge, or\_custom \\
6 & Free Generation: G30SPKI & llm\_judge, and\_custom \\
7 & Free Generation: Hijrah & llm\_judge, embedding\_similarity \\
8 & Free Generation: Adhan & llm\_judge, and\_custom \\
9 & Free Generation: Karak Kaliang & llm\_judge, or\_custom \\
10 & Free Generation: Giant Drum & llm\_judge\_drum \\
11 & Free Generation: Scientific Letter & llm\_judge\_science\_project \\
12 & Free Generation: School Life & llm\_judge\_aeni\_school\_life \\
13 & Free Generation: Olympics Medallists & llm\_judge \\
14 & Free Generation: Hillary Clinton & llm\_judge, embedding\_similarity,  \\
15 & Free Generation: MBZUAI & llm\_judge, and\_custom \\
16 & Free Generation: Grammar 

In [14]:
import os, json
from src import Questions

SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
questions = Questions().get_questions()

question_valid_ds = []
for q in questions:
    counter = 0
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as  ao:
            annotator_object = json.load(ao)
        if annotator_object["answers"][q]["answer"]:
            counter += 1
    question_valid_ds.append({
        "Task" : questions[q]["nickname"],
        "Valid answers" : counter,
    })
        

In [19]:
question_valid_ds_df = pd.DataFrame(question_valid_ds).drop(EXCLUDED_WHOLE)
print(question_valid_ds_df.to_latex(index=False))

\begin{tabular}{lr}
\toprule
Task & Valid answers \\
\midrule
Sentiment Analysis 3 Classes & 80 \\
Sentiment Analysis 2 Classes & 90 \\
Machine Translation & 92 \\
Dialogue Summarization & 97 \\
Topic Classification & 90 \\
Free Generation: Balonku Song & 93 \\
Free Generation: G30SPKI & 93 \\
Free Generation: Hijrah & 80 \\
Free Generation: Adhan & 84 \\
Free Generation: Karak Kaliang & 94 \\
Free Generation: Giant Drum & 90 \\
Free Generation: Scientific Letter & 83 \\
Free Generation: School Life & 97 \\
Free Generation: Olympics Medallists & 96 \\
Free Generation: Hillary Clinton & 98 \\
Free Generation: MBZUAI & 98 \\
Free Generation: Grammar Correction I & 98 \\
Free Generation: Grammar Correction II & 98 \\
Free Generation: Arithmetic & 99 \\
Free Generation: Shakespeare-Prompt Jailbreak & 97 \\
Multiple Choice Questions & 88 \\
Extractive QA & 91 \\
MultispanQA & 92 \\
Grammar Correctness Classification & 92 \\
Linguistically-conditioned Text Generation & 79 \\
\bottomrule
\end

In [16]:
custom_metrics = pd.read_csv("pfairness_questions - custom metric definition.csv")

custom_metrics.columns = custom_metrics.columns.str.replace('_', r'\\_', regex=True)
print(custom_metrics.replace('_', r'\\_', regex=True).replace('{', r'\{', regex=True).replace('}', r'\}', regex=True).to_latex(index=False))

\begin{tabular}{ll}
\toprule
metric name & description or prompt \\
\midrule
and\_custom & Given a model answer and a list of references, get the fuzzy string match score for each reference. Return the average. \\
or\_custom & Given a model answer and a list of references, get the fuzzy string match score for each reference. Return the maximum. \\
requirement\_checker & Given a set of rules, which consists only of minimum generation length and the existence of certain words, score based on how many rules an answer abides by. \\
llm\_judge & You are a judge that gives a score to a prediction answer based on how accurate it is compared to a reference answer.

Given this instruction right here:
\{prompt\}


On a scale of 0 (worst) to 1 (best), how would you rate the machine's answer?
Reference Answer: 
\{refs\}

Prediction Answer:
\{preds\}

Answer only the XML format of the score: <score>[float]</score> \\
llm\_judge\_drum & You are a judge that gives a holistic score to a text based on 